# 16 · 증분(전이) 마커 매핑 실험

**아이디어**: id0 기준으로 위치가 확정된 마커들이 늘어나면, 그 마커들을 기준으로 또 주변 마커를 등록 — **id0가 안 보이는 사진도 지도에 기여**하고, 지도가 바깥으로 자란다(전이 프로세스).

**알고리즘** (`src/mapping.py`, `build_map_incremental`):
1. **BFS 전이 성장**: 위치 확정 마커 ≥2개(부트스트랩은 seed 1개) 보이는 프레임 → 호모그래피 → 새 마커 등록. 웨이브마다 전 프레임 재평가. `hop` = 전이 단수.
2. **외삽 금지 게이트**: 각 프레임은 자기가 본 앵커들의 볼록껍질 + 마커 8칸 반경 안쪽 마커만 갱신 — 호모그래피는 관측 영역 밖(외삽)에서 오차 폭발.
3. **게이지 고정**: 호모그래피 지도는 전역 배율·기울임이 자유변수 → **모든 마커=물리 22mm** 제약으로 스케일 고정(변 길이 중앙값), seed로 원점·방향 고정.
4. **전역 반복 정제**: 공통 마커 전부로 재정합 반복(간이 번들 조정) → 등록 순서 의존성 완화.

**검증 데이터**: 실시간 스트리밍 세션 3개(자동 저장된 프레임). 기준 = 기존 `marker_map.json`(사진 19장 전역 뷰).

> ⚠️ 오차 해석 주의: "vs 기준지도"는 **두 지도 오차의 합**(기준지도도 완벽하지 않음 — 사진모드, 왜곡 미보정). 스트림 지도의 **내부 일관성은 ~1mm**로 수렴함.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import numpy as np, matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'; plt.rcParams['axes.unicode_minus'] = False
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import workspace as ws, mapping as mp, aruco_utils as au

REF, _, _, _ = ws.load_marker_map(os.path.join(ROOT, 'output', 'marker_map.json'))
K, dist = au.load_intrinsics(os.path.join(ROOT, 'output', 'phone_stream_intrinsics.npz'))
det = ws.make_detector()
S = os.path.join(ROOT, 'output', 'phone_stream')
sessions = sorted(glob.glob(os.path.join(S, 'session_*')))
print('세션:', [os.path.basename(s) for s in sessions])

# 관측 수집(왜곡 보정 포함) — 세션당 수십 초
obs_by = {}
for s in sessions:
    files = sorted(glob.glob(os.path.join(s, '*.jpg')))
    obs_by[os.path.basename(s)] = mp.detect_observations(files, det, K, dist)
names = list(obs_by)
o1, o2, o3 = (obs_by[n] for n in names[:3])

## 1) 데이터 특성 — 세션별 id0 포함율

세션2는 id0가 거의 없고 세션3은 **아예 없음** → 전이 없이는 지도 구축 불가능한 데이터.

In [ ]:
for n, obs in obs_by.items():
    cnt = [len(r) for r in obs]
    id0 = sum(1 for r in obs if 0 in r)
    print(f'{n}: {len(obs)}프레임 | 마커/장 평균 {np.mean(cnt):.1f} | id0 포함 {id0}/{len(obs)}장')

## 2) 실험 A — 전이 성립 검증 (id0 없는 데이터로 지도 구축)

세션2+3(id0 2/84장)만으로 31개 전부 등록되는지 + hop(전이 단수) 분포.

In [ ]:
def build_report(obs, tag):
    corners, stats, info = mp.build_map_incremental(obs)
    errs = mp.compare_maps(corners, REF); e = np.array(list(errs.values()))
    byhop = {}
    for i in errs:
        byhop.setdefault(stats.get(i, {}).get('hop', 99), []).append(errs[i])
    print(f'[{tag}] 등록 {len(corners)}/31, 미등록 {info["unregistered"]}')
    print(f'  vs 기준지도: 평균 {e.mean():.1f}mm 중앙 {np.median(e):.1f}mm 최대 {e.max():.1f}mm')
    for h in sorted(byhop):
        print(f'  hop {h}: {len(byhop[h])}개  평균 {np.mean(byhop[h]):.1f}mm  최대 {np.max(byhop[h]):.1f}mm')
    return corners, stats, byhop

c23, st23, bh23 = build_report(o2 + o3, '부분뷰만 (세션2+3, id0 2/84장)')

# 지도 시각화: 기준(회색) vs 구축(색=hop)
aligned, common = mp.align_rigid(c23, REF)
plt.figure(figsize=(7, 9))
for i in common:
    r = REF[i].mean(0); b = aligned[i].mean(0)
    h = st23.get(i, {}).get('hop', 9)
    plt.plot([r[0], b[0]], [r[1], b[1]], '-', color='red', alpha=.4)
    plt.scatter(*r, c='lightgray', s=90, zorder=1)
    plt.scatter(*b, c=['tab:blue', 'tab:green', 'tab:orange', 'tab:red'][min(h, 3)], s=35, zorder=2)
    plt.annotate(str(i), r, fontsize=7, ha='center', va='center')
plt.gca().invert_yaxis(); plt.axis('equal'); plt.grid(alpha=.3)
plt.title('기준지도(회색) vs 전이 구축(색=hop: 파랑0/초록1/주황2) — 빨간선=오차')
plt.show()

## 3) 실험 B — 전이 드리프트와 루프 클로저

hop이 늘수록 오차가 커지는가? 전역 뷰(세션1, 여러 마커를 한 프레임에)를 합치면 회복되는가?

In [ ]:
_, _, bh_all = build_report(o1 + o2 + o3, '전역뷰 포함 (세션1+2+3)')

hops = sorted(set(bh23) | set(bh_all))
x = np.arange(len(hops)); w = 0.35
plt.figure(figsize=(7, 4))
plt.bar(x - w/2, [np.mean(bh23.get(h, [np.nan])) for h in hops], w, label='부분뷰만(2+3)')
plt.bar(x + w/2, [np.mean(bh_all.get(h, [np.nan])) for h in hops], w, label='+전역뷰(1+2+3)')
plt.xticks(x, [f'hop {h}' for h in hops]); plt.ylabel('평균 오차 (mm)')
plt.title('전이 드리프트: hop별 오차 — 전역뷰(루프 클로저)가 hop 의존성을 제거')
plt.legend(); plt.grid(alpha=.3, axis='y'); plt.show()

## 4) 실험 C — 몇 장이면 되는가 (프레임 수 vs 커버리지·오차)

세션1의 앞 N장만으로 구축 → 등록 마커 수와 오차.

In [ ]:
Ns = [3, 5, 8, 12, 20, 30, len(o1)]
means, maxs, regs = [], [], []
for N in Ns:
    c, st, info = mp.build_map_incremental(o1[:N])
    errs = mp.compare_maps(c, REF); e = np.array(list(errs.values())) if errs else np.array([np.nan])
    means.append(np.nanmean(e)); maxs.append(np.nanmax(e)); regs.append(len(c))
    print(f'N={N:2d}: 등록 {len(c):2d}/31  평균 {np.nanmean(e):6.1f}mm  최대 {np.nanmax(e):6.1f}mm')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(Ns, means, 'o-', label='평균'); ax[0].plot(Ns, maxs, 's--', label='최대')
ax[0].set_xlabel('사용 프레임 수 N'); ax[0].set_ylabel('vs 기준지도 오차(mm)')
ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title('프레임 수 vs 오차')
ax[1].plot(Ns, regs, 'o-', color='tab:green'); ax[1].axhline(31, ls=':', color='gray')
ax[1].set_xlabel('사용 프레임 수 N'); ax[1].set_ylabel('등록 마커 수'); ax[1].grid(alpha=.3)
ax[1].set_title('등록 커버리지')
plt.tight_layout(); plt.show()

## 5) 결론 (실측 기반)

1. **전이 프로세스 성립** — id0가 2/84장(세션3은 0장)뿐인 부분 뷰 데이터만으로 **31/31 마커 전부 등록**. id0 없는 사진도 확정 마커를 기준으로 지도에 기여한다는 가설이 실증됨.
2. **전이 드리프트 실재** — 부분 뷰만 쓰면 hop이 늘수록 오차 증가(hop1 ~57mm → hop2 ~85mm). 사용자 직관대로 "주변→주변" 전이는 오차가 누적된다.
3. **루프 클로저(전역 뷰)가 해독제** — 여러 마커를 한 프레임에 담는 전역 뷰 세션을 합치면 hop 의존성이 사라짐(hop0/1/2 ≈ 31/30/26mm로 평탄). **매핑 시 전역 뷰 몇 장을 반드시 섞을 것.**
4. **몇 장이면 되는가** — 양질의 전역 뷰라면 **3~5장으로 이미 등록 완료 수준**(N=3에서 25/31, N=5에서 31/31). 그 이상은 커버리지가 아니라 평균화 효과이며, **저품질(블러·근접) 프레임이 섞이면 오히려 오차가 늘 수 있음**(N=30, 49에서 악화 관찰). → 장수보다 **품질과 시점 다양성**이 중요.
5. **정확도 한계(솔직)** — 스트림 프레임(1440×1920, 일부 블러) 지도는 기준지도와 ~20-35mm 불일치. 내부 일관성은 ~1mm까지 수렴하므로, 이 차이는 두 지도의 계통 오차(렌즈 왜곡 잔차·프레임 품질) 합. **정밀 지도는 고해상 사진으로, 스트림 매핑은 빠른 현장 셋업용**으로 역할 구분 권장.
6. **알고리즘 필수 요소**(제거 시 실패를 실측으로 확인):
   - 게이지 고정(마커 22mm 제약) 없으면 스케일 1.63배 폭주
   - 외삽 금지 게이트 없으면 소수 마커 프레임이 원거리 마커를 오염
   - 전역 정제 없으면(부분뷰) 전이 드리프트 그대로 잔존

**권장 매핑 프로토콜**: 전역 뷰(가능한 많은 마커가 한 프레임에) 3~5장 + 커버 안 되는 영역은 부분 뷰로 확장 + 떨어진 영역을 잇는 루프 클로저 컷 1~2장. 흔들림 없는 정지 상태에서 촬영.